# HistoQL — Historical Research Query Language

This notebook is the executable companion to TP1. It keeps two layers separate:

1. **HistoQL**, the readable frontend language used by historians.
2. **SQL**, a parameterised backend target generated only after parsing and semantic validation.

The SQL schema below is an explicit *demonstration adapter*, not part of the language definition. Changing a database should require changing only the adapter/compiler, never the HistoQL grammar.

## Supported surface syntax

```histoql
RESEARCH "Portuguese Liberalism" {
  LET politicians = FIND people
    WHO participated_in "Liberal Revolution"
    IN "Porto"
    DURING 1820..1834
    RETURN name, role;

  IF EXISTS politicians {
    FIND sources THAT mentions "Liberal Revolution" RETURN title, date;
  }
}
```

`LET`, `IF`, and `FOR EACH` meet the course requirements for declarations, scope, branching, and repetition. The query clauses (`WHO`/`THAT`, `IN`, `DURING`, `WHERE`, `RETURN`) express the historical domain.

In [13]:
from __future__ import annotations

import json
from dataclasses import dataclass
from typing import Any

from lark import Lark, Transformer, Token

class HistoQLSyntaxError(Exception):
    """The input does not conform to the context-free grammar."""

class HistoQLSemanticError(Exception):
    """The parsed program violates a HistoQL context condition."""


In [ ]:
# Context-free grammar (EBNF notation as accepted by Lark).
HISTOQL_GRAMMAR = r'''
start: research_block 
        | statement+

research_block: "RESEARCH" STRING "{" statement* "}"

?statement: let_stmt 
          | query_stmt 
          | if_stmt 
          | foreach_stmt

let_stmt: "LET" ID "=" query ";"

query_stmt: query ";"

if_stmt: "IF" condition "{" statement* "}" ["ELSE" "{" statement* "}"]

foreach_stmt: "FOR" "EACH" ID "IN" ID "{" statement* "}"

query: "FIND" entity clause* return_clause?

entity: PEOPLE 
        | GROUPS 
        | EVENTS 
        | PLACES 
        | INSTITUTIONS 
        | SOURCES
?clause: relation_clause 
        | place_clause 
        | during_clause 
        | where_clause

relation_clause: ("WHO" | "THAT") ID value
place_clause: "IN" STRING
during_clause: "DURING" year ".." year
where_clause: "WHERE" ID COMPOP value
return_clause: "RETURN" ID ("," ID)*

?condition: "EXISTS" ID                         -> exists
          | ID "COUNT" COMPOP INT               -> count_compare
?value: STRING | INT
year: INT

# prioridade máxima
PEOPLE.10: "people"
GROUPS.10: "groups"
EVENTS.10: "events"
PLACES.10: "places"
INSTITUTIONS.10: "institutions"
SOURCES.10: "sources"

COMPOP: "=" 
        | "!=" 
        | "<" 
        | "<=" 
        | ">" 
        | ">="

ID: /[a-zA-Z_][a-zA-Z0-9_]*/
%import common.ESCAPED_STRING -> STRING
%import common.INT
%import common.WS
%ignore WS
%ignore /#[^\n]*/
'''

parser = Lark(HISTOQL_GRAMMAR, parser="lalr", start="start")
print("HistoQL grammar loaded.")

HistoQL grammar loaded.


In [15]:
class ToAST(Transformer):
    def STRING(self, token): 
        return json.loads(str(token))
    
    def INT(self, token): 
        return int(token)
    
    def ID(self, token): 
        return str(token)
    
    def COMPOP(self, token): 
        return str(token)
    
    def year(self, items): 
        return items[0]
    
    def entity(self, items): 
        return str(items[0])
    
    def relation_clause(self, items): 
        return {"kind": "relation", "predicate": items[-2], "value": items[-1]}
    
    def place_clause(self, items): 
        return {"kind": "place", "value": items[0]}
    
    def during_clause(self, items): 
        return {"kind": "during", "start": items[0], "end": items[1]}
    
    def where_clause(self, items): 
        return {"kind": "where", "field": items[0], "operator": items[1], "value": items[2]}
    
    def return_clause(self, items): 
        return {"kind": "return", "fields": list(items)}
    
    def query(self, items):
        entity, *parts = items
        return {"kind": "query", "entity": entity, "clauses": [p for p in parts if p["kind"] != "return"], "return": next((p["fields"] for p in parts if p["kind"] == "return"), ["name"])}
    
    def let_stmt(self, items): 
        return {"kind": "let", "name": items[0], "query": items[1]}
    
    def query_stmt(self, items): 
        return {"kind": "query_stmt", "query": items[0]}
    
    def exists(self, items): 
        return {"kind": "exists", "name": items[0]}
    
    def count_compare(self, items): 
        return {"kind": "count_compare", "name": items[0], "operator": items[1], "value": items[2]}
    
    def if_stmt(self, items): 
        return {"kind": "if", "condition": items[0], "then": list(items[1:]), "else": []}
    
    def foreach_stmt(self, items): 
        return {"kind": "foreach", "item": items[0], "collection": items[1], "body": list(items[2:])}
    
    def research_block(self, items): 
        return {"kind": "research", "title": items[0], "statements": list(items[1:])}
    
    def start(self, items): 
        return items[0] if len(items) == 1 and items[0]["kind"] == "research" else {"kind": "program", "statements": list(items)}

def parse_histoql(source: str) -> dict[str, Any]:
    """Parse frontend HistoQL into a database-independent AST."""
    try:
        return ToAST().transform(parser.parse(source))
    except Exception as error:
        raise HistoQLSyntaxError(str(error)) from error


In [16]:
# Semantic model: attributes and legal relationships are language rules, not SQL table names.
ENTITY_FIELDS = {
    "people": {"name", "role", "occupation", "birth_year", "death_year"},
    "groups": {"name", "type", "start_year", "end_year"},
    "events": {"name", "date", "type", "start_year", "end_year"},
    "places": {"name", "type"},
    "institutions": {"name", "type", "start_year", "end_year"},
    "sources": {"title", "date", "type", "author"},
}
LEGAL_RELATIONS = {
    "people": {"participated_in", "lived_in", "worked_at", "ruled"},
    "groups": {"participated_in", "ruled"},
    "events": {"occurred_in", "before", "after"},
    "sources": {"mentions"},
    "institutions": {"located_in"},
    "places": set(),
}

class Scopes:
    def __init__(self): 
        self.frames: list[dict[str, str]] = [{}]

    def push(self): 
        self.frames.append({})

    def pop(self): 
        self.frames.pop()

    def declare(self, name: str, entity: str):
        if name in self.frames[-1]: raise HistoQLSemanticError(f"Duplicate declaration in this scope: {name}")
        self.frames[-1][name] = entity

    def lookup(self, name: str) -> str:
        for frame in reversed(self.frames):
            if name in frame: return frame[name]
        raise HistoQLSemanticError(f"Undeclared research result: {name}")

def validate_query(query: dict[str, Any]) -> None:
    entity = query["entity"]
    if entity not in ENTITY_FIELDS: raise HistoQLSemanticError(f"Unknown entity kind: {entity}")
    for field in query["return"]:
        if field not in ENTITY_FIELDS[entity]: raise HistoQLSemanticError(f"{field!r} is not a returnable field of {entity}")

    for clause in query["clauses"]:
        if clause["kind"] == "relation" and clause["predicate"] not in LEGAL_RELATIONS[entity]:
            raise HistoQLSemanticError(f"{clause['predicate']!r} is not valid for {entity}")
        
        if clause["kind"] == "during" and clause["start"] > clause["end"]:
            raise HistoQLSemanticError("A period must start on or before it ends")
        
        if clause["kind"] == "where" and clause["field"] not in ENTITY_FIELDS[entity]:
            raise HistoQLSemanticError(f"Unknown field {clause['field']!r} for {entity}")

def validate_program(node: dict[str, Any], scopes: Scopes | None = None) -> None:
    scopes = scopes or Scopes()
    statements = node.get("statements", [node])
    
    for statement in statements:
        if statement["kind"] == "let":
            validate_query(statement["query"]); scopes.declare(statement["name"], statement["query"]["entity"])

        elif statement["kind"] == "query_stmt": validate_query(statement["query"])

        elif statement["kind"] == "if":
            scopes.lookup(statement["condition"]["name"]); scopes.push(); validate_program({"statements": statement["then"]}, scopes); scopes.pop()
            
        elif statement["kind"] == "foreach":
            entity = scopes.lookup(statement["collection"]); scopes.push(); scopes.declare(statement["item"], entity); validate_program({"statements": statement["body"]}, scopes); scopes.pop()

# Entity existence and actual date availability require a historical catalogue/database;
# they are intentionally a later semantic phase, not properties of the grammar.

In [17]:
# Demonstration SQL adapter (PostgreSQL DB-API %s placeholders):
# entities(id, kind, name, role, occupation, birth_year, death_year, date, type, author, start_year, end_year, place_id)
# relationships(subject_id, predicate, object_id)
SQL_COLUMNS = set().union(*ENTITY_FIELDS.values())

def compile_query_to_sql(query: dict[str, Any]) -> tuple[str, list[Any]]:
    """Compile one validated FIND query. Values stay parameters; identifiers come from the validated language schema."""
    validate_query(query)
    select = ", ".join(f"e.{field}" for field in query["return"])
    joins, predicates, parameters = [], ["e.kind = %s"], [query["entity"]]

    for index, clause in enumerate(query["clauses"], start=1):
        if clause["kind"] == "relation":
            joins += [f"JOIN relationships r{index} ON r{index}.subject_id = e.id", f"JOIN entities o{index} ON o{index}.id = r{index}.object_id"]
            predicates += [f"r{index}.predicate = %s", f"o{index}.name = %s"]; parameters += [clause["predicate"], clause["value"]]

        elif clause["kind"] == "place":
            joins.append(f"JOIN entities p{index} ON p{index}.id = e.place_id"); predicates.append(f"p{index}.name = %s"); parameters.append(clause["value"])

        elif clause["kind"] == "during":
            predicates.append("e.start_year <= %s AND COALESCE(e.end_year, e.start_year) >= %s"); parameters += [clause["end"], clause["start"]]

        elif clause["kind"] == "where":
            predicates.append(f"e.{clause['field']} {clause['operator']} %s"); parameters.append(clause["value"])
            
    sql = f"SELECT {select} FROM entities e {' '.join(joins)} WHERE {' AND '.join(predicates)};"
    return sql, parameters


In [18]:
example = '''
RESEARCH "Portuguese Liberalism" {
  LET politicians = FIND people
    WHO participated_in "Liberal Revolution"
    IN "Porto"
    DURING 1820..1834
    RETURN name, role;
  IF EXISTS politicians {
    FIND sources THAT mentions "Liberal Revolution" RETURN title, date;
  }
}
'''

ast = parse_histoql(example)
validate_program(ast)
first_query = ast["statements"][0]["query"]
sql, parameters = compile_query_to_sql(first_query)
print("Semantic validation: OK")
print(sql)
print("parameters:", parameters)

# Suggested delivery checks:
# parse_histoql('FIND people DURING 1834..1820 RETURN name;')  -> syntactically valid, semantically invalid
# parse_histoql('LET x = FIND people RETURN unknown;')         -> syntactically valid, semantically invalid
# parse_histoql('FIND people RETURN name')                     -> syntactically invalid (missing semicolon)

Semantic validation: OK
SELECT e.name, e.role FROM entities e JOIN relationships r1 ON r1.subject_id = e.id JOIN entities o1 ON o1.id = r1.object_id JOIN entities p2 ON p2.id = e.place_id WHERE e.kind = %s AND r1.predicate = %s AND o1.name = %s AND p2.name = %s AND e.start_year <= %s AND COALESCE(e.end_year, e.start_year) >= %s;
parameters: ['people', 'participated_in', 'Liberal Revolution', 'Porto', 1834, 1820]
